In [1]:
import torch
torch.cuda.empty_cache()

from datasets import load_dataset
import json
import re
from transformers import Gemma3ForConditionalGeneration, AutoProcessor
from tqdm import tqdm

# Load dataset
medical_data = load_dataset("Ahmed-Selem/Shifaa_Arabic_Medical_Consultations")

# Initialize model and processor
model_name = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(model_name)
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0"
).eval()

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
sampling_params = {
    "max_new_tokens": 4080,
    "do_sample": True,
    "eos_token_id": processor.tokenizer.eos_token_id
}


In [3]:
def create_json_prompt_for_entity_annotation(text, **kwargs):
    attributes = {key: value for key, value in kwargs.items() if value != "n/a"}
    
    prompt = f"""
**Objective:**
Analyze the provided Arabic medical consultation text and identify named entities such as symptoms, diagnoses, medications, and medical procedures. Each entity should be meticulously labeled according to its type for straightforward extraction.

**Input Text:**
"{text}"

**Format Requirements:**
- The output should be formatted in JSON, containing the original text and a list of identified entities.
- Each entity in the text should be accurately marked and annotated in the 'entities' list.
- The text must remain unchanged; only annotate entities.
- Follow all listed attributes (e.g., language, medical_context).

**Entity Annotation Details:**
- Entity types can be multi-word, separated by spaces (e.g., "medical procedure").
- Entities can be nested within other entities.
- A single entity may have multiple types, listed in the "types" key.

**Output Schema:**
<start attribute_1="value1" attribute_2="value2" ...>
{{
  "text": "{text}",
  "entities": [
    {{"entity": "entity name", "types": ["type 1", "type 2", ...]}},
    ...
  ]
}}
<end>

**Example:**
<start language="arabic" medical_context="consultation">
{{
  "text": "المريض يعاني من ارتفاع في ضغط الدم وألم في الصدر. تم وصف أتينولول وإجراء فحص تخطيط القلب.",
  "entities": [
    {{"entity": "ارتفاع في ضغط الدم", "types": ["diagnosis"]}},
    {{"entity": "ألم في الصدر", "types": ["symptom"]}},
    {{"entity": "أتينولول", "types": ["medication"]}},
    {{"entity": "فحص تخطيط القلب", "types": ["medical procedure"]}}
  ]
}}
<end>
"""
    attributes_string = " ".join([f'{key}="{value}"' for key, value in attributes.items()])
    prompt += f"<start {attributes_string}>"
    return prompt

# Generation function to annotate entities
def annotate_text(text, **kwargs):
    if not text or not isinstance(text, str):
        print("Invalid input text:", text)
        return None
    
    # Create prompt for this specific text
    prompt = create_json_prompt_for_entity_annotation(text, **kwargs)
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant for medical text annotation."}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ]
    
    try:
        # Process with chat template
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda:0", dtype=torch.bfloat16)
        
        # Debug: Inspect inputs
        print("Input text (first 50 chars):", text[:50])
        print("Input IDs (first 10):", inputs["input_ids"].tolist()[0][:10])
        print("Input IDs shape:", inputs["input_ids"].shape)
        
        # Validate token IDs
        vocab_size = len(processor.tokenizer)
        if (inputs["input_ids"] >= vocab_size).any() or (inputs["input_ids"] < 0).any():
            print("Invalid token IDs detected:", inputs["input_ids"])
            return None
        
        # Generate
        with torch.inference_mode():
            outputs = model.generate(**inputs, **sampling_params)
        input_len = inputs["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)
        
        # Extract JSON from the generated text
        json_start = generated_text.find("{")
        json_end = generated_text.rfind("}") + 1
        if json_start == -1 or json_end == 0:
            print("No valid JSON found in output:", generated_text[:100])
            return None
        json_str = generated_text[json_start:json_end]
        return json.loads(json_str)
    except Exception as e:
        print(f"Error during processing: {e}")
        return None


In [4]:
def tokenize_text(text):
    """Tokenize the input text into a list of tokens."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

def extract_entities(data):
    all_examples = []
    
    for dt in data:
        if not dt:
            continue
        try:
            tokens = tokenize_text(dt['text'])
            ents = [(k["entity"], k["types"]) for k in dt['entities']]
        except:
            continue

        spans = []
        for entity in ents:
            entity_tokens = tokenize_text(str(entity[0]))
            for i in range(len(tokens) - len(entity_tokens) + 1):
                if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                    for el in entity[1]:
                        spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ')))
        
        all_examples.append({"tokenized_text": tokens, "ner": spans})
    
    return all_examples

In [5]:

# Process the dataset with tqdm
generated_data = []


In [6]:
sample_data = medical_data["train"].shuffle(seed=42).select(range(3))

In [7]:

for example in tqdm(sample_data, desc="Annotating dataset"):
    text = example['Answer']
    output = annotate_text(
        text,
        language="arabic",
        medical_context="consultation"
    )
    if output:
        generated_data.append(output)


Annotating dataset:   0%|          | 0/3 [00:00<?, ?it/s]

Input text (first 50 chars): بسم الله الرحمن الرحيم
الأخت الفاضلة/ إيني حفظها ا
Input IDs (first 10): [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 573]
Input IDs shape: torch.Size([1, 1807])


Annotating dataset:  33%|███▎      | 1/3 [03:25<06:51, 205.83s/it]

Input text (first 50 chars): بسم الله الرحمن الرحيم
الأخت الفاضلة/ jinan حفظها 
Input IDs (first 10): [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 573]
Input IDs shape: torch.Size([1, 544])


Annotating dataset:  67%|██████▋   | 2/3 [06:49<03:24, 204.80s/it]

Input text (first 50 chars): بسم الله الرحمن الرحيم
الأخ الفاضل/ أحمد حفظه الله
Input IDs (first 10): [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 573]
Input IDs shape: torch.Size([1, 2734])


Annotating dataset: 100%|██████████| 3/3 [10:15<00:00, 205.30s/it]

Error during processing: Extra data: line 20 column 1 (char 4040)


In [8]:
sample_data['Answer']

['بسم الله الرحمن الرحيم\nالأخت الفاضلة/ إيني حفظها الله.\nالسلام عليكم ورحمة الله وبركاته، وبعد:\n\nإن الإفرازات المهبلية ذات الرائحة الكريهة تدل على وجود التهابات جرثومية, وغالبا ما تكون من النوع المختلط، أي الذي يشترك في إحداثه أكثر من جرثوم، والأفضل دوما أخذ عينة من هذه الإفرازات, لعمل زراعة لها، وتحديد أفضل مضاد حيوي يقضي على هذه الجراثيم، لكن ولمساعدتك الآن, أنصحك أن تتناولي العلاج التالي:\n\n1- حبوبا تسمى كلينداميسين clindamycin عيار 300 ملغ، حبة صباحا وحبة مساء لمدة أسبوع.\n\n2- بعد يومين من انتهاء العلاج السابق، يمكنك تناول حبة واحدة فقط من دواء يسمى دفلوكان deflucan عيار 150 ملغ.\n\n3- طوال فترة العلاج بالحبوب السابقة، يجب عليك استخدام كريم يسمى كيناكومب kenacomb، ثلاث مرات على الفرج.\n\nبالنسبة لما شاهدته الطبيبة من حبيبات على المبيض, فهي على الأغلب الأجربة التي تحتوي على البويضات, فإن كانت الدورة الشهرية عندك منتظمة، ولم يكن لديك شكوى من أعراض ارتفاع هرمون الذكورة مثل: ظهور شعر زائد في أماكن غير مألوفة, تساقط شديد في شعر الرأس, ظهور حبوب في الوجه والصدر والظهر, السمنة الجذع

In [16]:
# len(generated_data)
generated_data[1]

{'text': 'بسم الله الرحمن الرحيم\nالأخت الفاضلة/ jinan حفظها الله.\nالسلام عليكم ورحمة الله وبركاته وبعد،،،\n\nارتفاع هرمون البرولكتين يؤدي إلى زيادة في حجم الثدي.\n\nوالله الموفق.',
 'entities': [{'entity': 'ارتفاع هرمون البرولكتين', 'types': ['diagnosis']},
  {'entity': 'زيادة في حجم الثدي', 'types': ['symptom']}]}

In [10]:
processed_output = extract_entities(generated_data)

# Save to JSON for training
def save_data_to_file(data, filepath):
    """Saves the processed data to a JSON file."""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)

output_file = "medical_data_gliner.json"
save_data_to_file(processed_output, output_file)

# Basic statistics
lengths = [len(d["tokenized_text"]) for d in processed_output]
print("Avg num tokens:", sum(lengths) / len(lengths) if lengths else 0)

len_ner = [len(d["ner"]) for d in processed_output]
print("Avg num of entities:", sum(len_ner) / len(len_ner) if len_ner else 0)

unique_entities = [n[2].lower() for d in processed_output for n in d["ner"]]
print("Unique entity types:", len(set(unique_entities)))

from collections import Counter
print("Top 5 entity types:", Counter(unique_entities).most_common(5))

Avg num tokens: 225.0
Avg num of entities: 15.0
Unique entity types: 7
Top 5 entity types: [('medication', 9), ('symptom', 8), ('diagnosis', 3), ('medical procedure', 3), ('body part', 3)]
